## 1. Problem
Cardiotocography (CTG) records the fetal heart rate and uterine contractions during pregnancy, and obstetricians use it to judge whether the fetus is in distress. Reading a CTG is partly subjective, so automated decision support could give less experienced staff a second opinion. This project uses the public UCI Cardiotocography dataset (2,126 recordings) to predict the fetal state (normal, suspect or pathological) from the measured CTG features. The main question is: how well do standard classifiers reproduce the experts' consensus label, and how reliably do they detect the rare pathological class compared with a majority-class baseline? The finished project delivers a comparison of at least three models with class-wise metrics and an error analysis. It is a course project and not a diagnostic tool.

## 2. Data

- **Source:** UCI Machine Learning Repository, Cardiotocography, https://archive.ics.uci.edu/dataset/193/cardiotocography
- **Size:** 2,126 recordings, 21 CTG features, expert labels
- **Licence:** CC BY 4.0
- **Loading:** downloaded automatically with the `ucimlrepo` package (dataset id 193) in the Setup section below.

### Data note

1. **Who:** 2,126 fetal CTG recordings. Only pregnancies that were monitored with CTG are included; fetuses that were not monitored could never be in the data. Mother's age, gestational week and other patient details are not included, and it is unknown whether several recordings come from the same mother.
2. **Why:** The recordings were processed automatically (SisPorto software) and classified by experts to develop and evaluate CTG analysis. They were not collected for this prediction task.
3. **When and where:** Portugal. TODO: check the original paper (Ayres de Campos et al., 2000) for years and sites; they are not stated on the UCI page.
4. **Label:** `NSP` = fetal state: 1 Normal, 2 Suspect, 3 Pathological. It is the consensus of three expert obstetricians, so it is the experts' reading of the trace, not a measured clinical outcome. Class shares are about 78% / 14% / 8% (to be confirmed in Section 4).
5. **Missing:** UCI reports no missing values, so MCAR/MAR/MNAR does not apply directly. Section 4 checks whether missing values are hidden as zeros.
6. **Impossible values:** To be checked in Section 4 (e.g. heart-rate baseline `LB`, `Min`/`Max` and histogram columns outside plausible ranges). Zeros in count columns such as decelerations can be legitimate.
7. **Timing / leakage:** All features come from the same recording that the experts read, so the label is closely tied to the features. The 10-class morphologic pattern column `CLASS` comes from the same expert reading and is **excluded** as a feature to avoid leakage.
8. **Consent:** Not stated on the UCI page. The data are public, anonymised numeric features without identifiers. TODO: check the original paper for an ethics statement.

## 3. Setup

Install the one package that is not in Colab by default, import the libraries, fix the random seed and load the data.


In [ ]:
# Import necessary libraries
!pip install -q ucimlrepo

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

SEED = 42
np.random.seed(SEED)

In [9]:
# Load the dataset from UCI Machine Learning Repository
ctg = fetch_ucirepo(id=193)
X = ctg.data.features
y = ctg.data.targets
print(y.columns.tolist())

df = X.copy()
df["NSP"] = y["NSP"]   # 1 = Normal, 2 = Suspect, 3 = Pathological
# CLASS (10-class morphologic pattern) is deliberately NOT used as a feature (leakage)

# Save a local copy
os.makedirs("data", exist_ok=True)
df.to_csv("data/ctg.csv", index=False)

print(df.shape)
df["NSP"].value_counts().sort_index()

['CLASS', 'NSP']
(2126, 22)


NSP
1    1655
2     295
3     176
Name: count, dtype: int64